ساختن ابزار های مورد برای ایجنت
و پیس پردازش برای تنظیم تایپ اعداد

In [79]:
import pandas as pd

class BankingDataManager:

    def __init__(
        self,
        customers_path,
        orders_path,
        requests_path
    ):

        self.customers_path = customers_path
        self.orders_path = orders_path
        self.requests_path = requests_path

        self.customers = pd.read_excel(
            customers_path
        )

        self.orders = pd.read_excel(
            orders_path
        )
        self.CUSTOMER_COLUMNS = {
            "کد ملی": "national_id",
            "نام و نام خانوادگی": "name",
            "شماره تماس": "phone",
            "موجودی حساب": "balance",
            "امتیاز اعتباری": "credit_score",
            "تاریخ عضویت": "join_date",
            "تعداد تراکنش‌های ماه اخیر": "monthly_transactions",
        }

        self.ORDER_COLUMNS = {
            "کد ملی": "national_id",
            "نوع کارت": "card_type",
            "تاریخ سفارش": "order_date",
            "وضعیت": "status",
            "کد پیگیری": "tracking_code",
            "تاریخ تحویل تقریبی": "estimated_delivery",
            "آدرس ارسال": "shipping_address",
        }

        self.normalize()

    def normalize(self):
      self.customers = self.customers.rename(columns=self.CUSTOMER_COLUMNS)
      self.orders = self.orders.rename(columns=self.ORDER_COLUMNS)
      self.customers["national_id"] = (
        self.customers["national_id"]
        .astype(str)
        .str.strip()
      )
      self.orders["national_id"] = (
          self.orders["national_id"]
          .astype(str)
          .str.strip()
      )
      self.customers["national_id"] = (
          self.customers["national_id"]
          .str.replace(r"\.0$", "", regex=True)
          .str.zfill(10)
      )
      self.orders["tracking_code"] = (
          self.orders["tracking_code"]
          .astype(str)
          .str.replace(r"\.0$", "", regex=True)
          .str.strip()
      )

    def get_customer(self, national_id):

      """
      Return customer information by national ID.
      """

      national_id = str(national_id).strip().zfill(10)

      result = self.customers[
          self.customers["national_id"] == national_id
      ]

      if result.empty:
          return None

      return result.iloc[0].to_dict()

    def get_orders(self, national_id: str):
      """
      Return all card orders for a customer.
      """

      national_id = str(national_id).strip().zfill(10)
      print(national_id)
      result = self.orders[
          self.orders["national_id"] == national_id
      ]
      print(result)

      if result.empty:
          return []

      return result.to_dict(orient="records")

    def get_active_orders(self, national_id: str):
      """
      Return currently active card orders.
      """
      ACTIVE_STATUSES = {
        "در دست بررسی",
        "تأیید شده",
        "در حال چاپ",
        "ارسال به پست",
        "ارسال شده",
      }

      customer_orders = self.get_orders(national_id)
      print(customer_orders)
      return [
          order
          for order in customer_orders
          if order["status"] in ACTIVE_STATUSES
      ]


    def get_order_history(self,
          national_id: str,
          delivered_only: bool = False
      ):
      """
      Return customer's card order history.
      """

      customer_orders = self.get_orders(national_id)

      if delivered_only:
          customer_orders = [
              order
              for order in customer_orders
              if order["status"] == "تحویل داده شده"
          ]

      customer_orders.sort(
          key=lambda x: x["order_date"]
          if x["order_date"] is not None
          else pd.Timestamp.min,
          reverse=True
      )

      return customer_orders

    def create_request(
      self,
      national_id,
      request_type,
      description
      ):
      import os
      from datetime import datetime

      REQUEST_COLUMNS = [
          "national_id",
          "name",
          "request_type",
          "description",
          "date",
          "status",
      ]

      national_id = (
          str(national_id)
          .strip()
          .replace(".0", "")
          .zfill(10)
      )

      # 1. Validate customer
      customer = self.get_customer(national_id)

      if customer is None:
          raise ValueError(
              "Customer with this national ID does not exist."
          )

      # 2. Get customer name
      name = customer["name"]

      # 3. Current date/time
      now = datetime.now()

      # 4. Load existing requests
      if os.path.exists(self.requests_path):
          request_df = pd.read_excel(self.requests_path)

          # Make sure required columns exist
          missing_columns = [
              col for col in REQUEST_COLUMNS
              if col not in request_df.columns
          ]

          if missing_columns:
              raise ValueError(
                  f"requests.xlsx is missing columns: {missing_columns}"
              )

      else:
          request_df = pd.DataFrame(columns=REQUEST_COLUMNS)

      # 5. Normalize existing data before duplicate check
      if not request_df.empty:

          request_df["national_id"] = (
              request_df["national_id"]
              .astype(str)
              .str.strip()
              .str.replace(r"\.0$", "", regex=True)
              .str.zfill(10)
          )

          request_df["date"] = pd.to_datetime(
              request_df["date"],
              errors="coerce"
          )

          # 6. Duplicate check
          today = now.date()

          duplicate = request_df[
              (request_df["national_id"] == national_id)
              &
              (request_df["request_type"] == request_type)
              &
              (request_df["date"].dt.date == today)
          ]

          if not duplicate.empty:
              return {
                  "success": False,
                  "duplicate": True,
                  "message": "این درخواست امروز قبلاً ثبت شده است."
              }

      # 7. Create new request
      new_request = {
          "national_id": national_id,
          "name": name,
          "request_type": request_type,
          "description": description,
          "date": now,
          "status": "ثبت شده",
      }

      # 8. Add request
      request_df = pd.concat(
          [
              request_df,
              pd.DataFrame([new_request])
          ],
          ignore_index=True
      )

      # 9. Save
      request_df.to_excel(
          self.requests_path,
          index=False
      )

      return {
          "success": True,
          "duplicate": False,
          "request": new_request
      }

In [2]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    langchain-openai \
    langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
!pip install -q pandas openpyxl python-docx sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 7.9 MB/s eta 0:00:00


پیش پردازش قوانین بانک

In [40]:
from docx import Document


def load_policy_document(path):
    doc = Document(path)

    parts = []

    for paragraph in doc.paragraphs:
        text = paragraph.text.strip()

        if text:
            parts.append(text)

    for table in doc.tables:
        for row in table.rows:
            cells = [
                cell.text.strip()
                for cell in row.cells
            ]

            if any(cells):
                parts.append(" | ".join(cells))

    return "\n".join(parts)

In [8]:
from google.colab import files
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}"')

Saving card_orders(3).xlsx to card_orders(3).xlsx
Saving customers.xlsx to customers.xlsx
Saving قوانین و مقررات بانک دایا_تدبیر.docx to قوانین و مقررات بانک دایا_تدبیر.docx
User uploaded file "card_orders(3).xlsx"
User uploaded file "customers.xlsx"
User uploaded file "قوانین و مقررات بانک دایا_تدبیر.docx"


In [7]:
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [5]:
!ls


sample_data


In [9]:
policy_text = load_policy_document(
    "قوانین و مقررات بانک دایا_تدبیر.docx"
)

print(policy_text)

قوانین و مقررات بانک دایا‌تدبیر
نسخه 2.1 | آذر 1404
1. انواع حساب‌های بانکی
1.1 حساب جاری
موجودی اولیه: حداقل 500,000 تومان
کارمزد ماهانه: رایگان
تعداد تراکنش رایگان: نامحدود
امکان چک: دارد
سود: ندارد
مناسب برای: کسب‌وکارها و پرداخت‌های روزانه
1.2 حساب پس‌انداز کوتاه‌مدت
موجودی اولیه: حداقل 1,000,000 تومان
نرخ سود سالانه: 18%
برداشت: بدون محدودیت
کارمزد: رایگان
مناسب برای: پس‌انداز روزمره
1.3 حساب پس‌انداز بلندمدت
موجودی اولیه: حداقل 5,000,000 تومان
نرخ سود سالانه: 22%
مدت زمان: حداقل 6 ماه
جریمه برداشت زودهنگام: 3% از سود
مناسب برای: پس‌انداز بلندمدت و سرمایه‌گذاری
1.4 حساب سرمایه‌گذاری ویژه
موجودی اولیه: حداقل 50,000,000 تومان
نرخ سود سالانه: 25%
مدت زمان: حداقل 12 ماه
مشاوره رایگان: دارد
مدیر حساب اختصاصی: دارد
مناسب برای: سرمایه‌گذاران حرفه‌ای
2. انواع کارت‌های بانکی
2.1 کارت نقدی استاندارد
ویژگی‌های کلی:
کارمزد صدور: 200,000 تومان
کارمزد سالانه: 150,000 تومان
زمان صدور: 5-7 روز کاری
اعتبار: 4 سال
محدودیت‌های برداشت:
حداکثر برداشت روزانه از دستگاه خودپرداز: 20,000,000 تومان
حداکثر 

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/colab_lib')


Mounted at /content/drive


In [11]:
!ls


'card_orders(3).xlsx'   drive	     'قوانین و مقررات بانک دایا_تدبیر.docx'
 customers.xlsx         sample_data


In [80]:

db = BankingDataManager(
    customers_path="/content/customers.xlsx",
    orders_path="/content/card_orders(3).xlsx",
    requests_path="/content/requests.xlsx"
)

In [13]:
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

splitting and chunking

In [42]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
)

chunks = splitter.create_documents(
    [policy_text]
)

لود مدل embedding

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

ساخت vector database

In [17]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

/tmp/ipykernel_3315/3106790601.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [18]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3
    }
)

In [ ]:
docs = retriever.invoke(
    "شرایط دریافت وام چیست؟"
)

Langchain Tools

In [19]:
from langchain_core.tools import tool


In [20]:
@tool
def authenticate_customer(national_id: str) -> dict:
    """
    Authenticate a customer using their national ID.

    Use this before accessing customer-specific information.
    """

    customer = db.get_customer(national_id)

    if customer is None:
        return {
            "authenticated": False,
            "message": "مشتری با این کد ملی در سیستم پیدا نشد."
        }

    return {
        "authenticated": True,
        "national_id": customer["national_id"],
        "name": customer["name"]
    }

In [21]:


@tool

def search_bank_policy(query: str) -> str:
    """
    Search the bank policy document for information
    relevant to a user's policy question.
    """

    docs = retriever.invoke(query)

    if not docs:
        return "No relevant policy information was found."

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [22]:
@tool
def get_customer(national_id: str) -> dict:
    """
    Get customer information such as balance and credit score.

    Use this when a question requires customer-specific financial
    information.
    """

    customer = db.get_customer(national_id)

    if customer is None:
        return {
            "found": False,
            "message": "مشتری پیدا نشد."
        }

    return {
        "found": True,
        "customer": customer
    }

In [23]:
@tool
def get_orders(national_id: str) -> list:
    """
    Get all card orders for a customer.
    """

    return db.get_orders(national_id)

In [24]:
@tool
def get_active_orders(national_id: str) -> dict:
    """
    Get all currently active card orders for a customer.

    Use this for questions about:
    - card status
    - where the card is
    - whether the card has arrived
    - when the card will arrive
    """

    orders = db.get_active_orders(national_id)

    return {
        "count": len(orders),
        "orders": orders
    }

In [25]:
@tool
def get_order_history(national_id: str) -> dict:
    """
    Get the customer's card order history.

    Use this when the user asks about:
    - previous cards
    - last delivered card
    - historical orders
    """

    orders = db.get_order_history(national_id)

    return {
        "count": len(orders),
        "orders": orders
    }

In [26]:
@tool
def create_request(
    national_id: str,
    request_type: str,
    description: str
) -> dict:
    """
    Create a customer request.

    Request types:
    - وام
    - افتتاح حساب
    - سفارش کارت جدید
    - شکایت
    - استعلام
    - سایر
    """

    return db.create_request(
        national_id=national_id,
        request_type=request_type,
        description=description
    )

In [27]:
@tool
def calculate_order_age(order_date: str) -> dict:
    """
    Calculate how many calendar days have passed since
    a card order date.
    """

    order_date = pd.to_datetime(
        order_date,
        errors="coerce"
    )

    if pd.isna(order_date):
        return {
            "success": False,
            "message": "تاریخ سفارش نامعتبر است."
        }

    today = pd.Timestamp.now().normalize()

    days = (today - order_date.normalize()).days

    return {
        "success": True,
        "days_elapsed": days
    }

In [29]:
SYSTEM_PROMPT = """
تو یک Agent بانکی هوشمند هستی که به زبان فارسی با مشتری صحبت می‌کنی.

وظایف اصلی:
- شناسایی مشتری با کد ملی
- پیگیری سفارش کارت
- پاسخ به سوالات قوانین و سیاست‌های بانک
- پاسخ به سوالات ترکیبی با استفاده از چند منبع
- ثبت درخواست مشتری

قوانین مهم:

1. برای اطلاعات شخصی مشتری هرگز حدس نزن.
   اطلاعات مشتری فقط باید از ابزارهای مربوط به customer دریافت شود.

2. برای اطلاعات سفارش کارت از ابزارهای سفارش کارت استفاده کن.

3. برای قوانین بانک، شرایط وام، انواع کارت، زمان صدور کارت،
   محدودیت برداشت و سایر policyها از search_bank_policy استفاده کن.

4. اگر سوال نیاز به چند منبع دارد، تمام منابع لازم را بررسی کن.
   مثلا برای بررسی امکان دریافت وام:
   - اطلاعات مشتری
   - policy وام
   هر دو لازم هستند.

5. اگر کاربر چند سفارش کارت فعال دارد، همه سفارش‌ها را بررسی کن
   و اگر لازم بود از کاربر بپرسد کدام کارت را منظور دارد.

6. اگر سفارش فعال وجود ندارد، این موضوع را واضح به کاربر بگو.

7. اگر سوال به سفارش قبلی و تحویل‌شده مربوط است،
   از تاریخچه سفارش‌ها استفاده کن.

8. برای محاسبات تاریخ و تعداد روزها به نتیجه ابزارها اعتماد کن
   و تاریخ را خودت حدس نزن.

9. قبل از ثبت درخواست، هویت مشتری را بررسی کن.

10. هنگام ثبت درخواست، نوع درخواست و توضیحات مناسب را مشخص کن.

11. اگر اطلاعات لازم برای پاسخ وجود ندارد، سوال روشن‌کننده بپرس.

12. پاسخ نهایی را کوتاه، واضح و به زبان فارسی ارائه کن.

13. اطلاعاتی مثل کد پیگیری، موجودی، امتیاز اعتباری،
    آدرس و تاریخ را فقط از داده واقعی ابزارها نمایش بده.

14. هرگز به کاربر نگو که اطلاعاتی را از خودت حدس زده‌ای.

انواع درخواست:
- وام
- افتتاح حساب
- سفارش کارت جدید
- شکایت
- استعلام
- سایر
"""

In [30]:
tools = [
    authenticate_customer,
    get_customer,
    get_orders,
    get_active_orders,
    get_order_history,
    search_bank_policy,
    create_request,
]

In [ ]:
state = {
    "national_id": None,
    "last_intent": None,
    "last_entity": None,
    "conversation": []
}

In [ ]:
!pip show langchain langchain-core langchain-community

Name: langchain
Version: 1.4.3
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-core
Version: 1.6.6
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: httpx, jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-classic, langchain-community, langchain-huggingface, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk
---
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
Li

In [31]:
from langchain.agents import create_agent

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)


In [32]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),

    MessagesPlaceholder(
        variable_name="chat_history"
    ),

    ("human", "{input}"),

    MessagesPlaceholder(
        variable_name="agent_scratchpad"
    )
])

In [33]:
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_history = InMemoryChatMessageHistory()

/tmp/ipykernel_3315/2835580824.py:3: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  chat_history = InMemoryChatMessageHistory()


RAG Implementation

In [50]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
    extra_body={
        "reasoning": {
            "enabled": True
        }
    }
)

In [52]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

تست پایه

In [86]:
def chat(user_message):

    chat_history.add_user_message(user_message)

    result = agent.invoke({
        "messages": chat_history.messages
    })

    answer = result["messages"][-1].content

    chat_history.add_ai_message(answer)

    return answer
print(chat('سلام، کد ملی من 0123456789 است. کارتم چی شد؟"'))

KeyboardInterrupt: 

In [54]:
print(chat('یه کارت اعتباری جدید میخوام ، چقدر طول میکشه'))

برای صدور یک کارت اعتباری جدید معمولاً زمان پردازش و چاپ کارت بین **۵ تا ۷ روز کاری** است. این بازه زمانی بر اساس سیاست‌های بانک برای کارت‌های استاندارد (نقدی) تعریف شده و برای کارت‌های اعتباری نیز به‌طور معمول مشابه است.

اگر می‌خواهید درخواست صدور کارت اعتباری را ثبت کنید، لطفاً کد ملی خود را ارائه دهید تا هویت شما تأیید شود و سپس درخواست شما را در سیستم ثبت کنیم.


Card Order Tracking دقت در پیگیری و نمایش وضعیت کارت

In [60]:
print(chat('کد ملی من 0221536568وضعیت کارتم چیه '))

0221536568
Empty DataFrame
Columns: [national_id, card_type, order_date, status, tracking_code, estimated_delivery, shipping_address]
Index: []
[]
0221536568
Empty DataFrame
Columns: [national_id, card_type, order_date, status, tracking_code, estimated_delivery, shipping_address]
Index: []
سلام مهدی جعفری عزیز،

بررسی کردم و در سیستم شما **هیچ سفارش کارت فعال** یا **تاریخچه‌ای از سفارش‌های قبلی** وجود ندارد. به عبارت دیگر، در حال حاضر هیچ کارت در حال پردازش یا تحویل نیست و تا به حال هم هیچ کارتی برای شما سفارش نشده است.

اگر می‌خواهید یک کارت جدید (اعتباری، debit یا پیش‌اپی) سفارش دهید، لطفاً بگویید تا هویت شما را تأیید کنم و درخواست شما را ثبت کنم.


In [61]:
print(chat('سلام کد ملی من 8395591067 وضعیت سفارشم چیشد'))

8395591067
  national_id card_type  order_date        status tracking_code  \
1  8395591067      هدیه  1404/10/05  در دست بررسی    9496770302   

   estimated_delivery                                   shipping_address  
1  1404/10/14 (تأخیر)  قم، نارمک، خیابان امام خمینی، کوچه 5، پلاک 33،...  
[{'national_id': '8395591067', 'card_type': 'هدیه', 'order_date': '1404/10/05', 'status': 'در دست بررسی', 'tracking_code': '9496770302', 'estimated_delivery': '1404/10/14 (تأخیر)', 'shipping_address': 'قم، نارمک، خیابان امام خمینی، کوچه 5، پلاک 33، واحد 16'}]
سلام آیدا میرزایی عزیز،

سفارش کارت هدیه شما در حال **بررسی** است.  
- تاریخ سفارش: ۱۴۰۴/۱۰/۰۵  
- کد پیگیری: ۹۴۹۶۷۷۰۳۰۲  
- تاریخ تحویل پیش‑بینی شده: ۱۴۰۴/۱۰/۱۴ (با تأخیر)  
- آدرس ارسال: قم، نارمک، خیابان امام خمینی، کوچه ۵، پلاک ۳۳، واحد ۱۶

اگر سؤال دیگری دارید یا می‌خواهید وضعیت را بعداً بررسی کنم، بگویید.


Hybrid Queries ترکیب صحیح چند منبع داده

In [62]:
print(chat('سلام کد ملی من 8858226637 آیا میتونم وام بگیرم؟'))

مهسا میرزایی عزیز، بر اساس اطلاعات شما:

- امتیاز اعتباری: **372**  
- موجودی حساب: حدود ۱۰،۱۵۵،۸۲۷ تومان  
- تاریخ عضویت: ۱۳۹۲/۰۴/۱۲ (اکثر از ۲ سال)

برای دریافت **وام قرض‌الحسنه** (و انواع وام‌های دیگر مانند خودرو و مسکن) حداقل امتیاز اعتباری مورد نیاز **۷۰۰** (برای وام قرض‌الحسنه) یا **۶۵۰‑۷۰۰** (برای وام خودرو/مسکن) است. امتیاز شما به‌طور قابل‌توجهی تحت این آستانه است، بنابراین در حال حاضر **شما برای دریافت وام از بانک qualificated نیستید**.

برای بهبود شرایط، می‌توانید به افزایش امتیاز اعتباری (مثلاً از طریق پرداخت به‌موقع بدهی‌ها و کاهش استفاده از سقف اعتبار) و داشتن تاریخچه حساب ثابت و بدون چک برگشتی کار کنید. اگر سؤال دیگری دارید یا می‌خواهید در مورد بهبود امتیاز اعتباری راهنمایی بگیریم، خوشحال می‌شوم کمک کنم.


In [63]:
print(chat('چرا کارتم هنوز نیومده 7990475355  ؟'))

7990475355
   national_id card_type  order_date          status tracking_code  \
0   7990475355      نقدی  1404/10/05    در دست بررسی     135712923   
24  7990475355      نقدی  1404/09/25       ارسال شده    1011520871   
26  7990475355   اعتباری  1404/09/21  تحویل داده شده    2182114766   

    estimated_delivery                                   shipping_address  
0   1404/10/19 (تأخیر)  تهران، یوسف آباد، خیابان رسالت، کوچه 26، پلاک ...  
24          1404/10/05  قم، فرمانیه، خیابان کارگر، کوچه 5، پلاک 238، و...  
26          1404/09/28  تهران، نیاوران، خیابان انقلاب، کوچه 3، پلاک 6،...  
[{'national_id': '7990475355', 'card_type': 'نقدی', 'order_date': '1404/10/05', 'status': 'در دست بررسی', 'tracking_code': '135712923', 'estimated_delivery': '1404/10/19 (تأخیر)', 'shipping_address': 'تهران، یوسف آباد، خیابان رسالت، کوچه 26، پلاک 216، واحد 18'}, {'national_id': '7990475355', 'card_type': 'نقدی', 'order_date': '1404/09/25', 'status': 'ارسال شده', 'tracking_code': '1011520871', 'estimat

پیاده چت همراه با نشون دادن مراحل tool calling و reasoning

In [64]:
def chat(user_message):

    chat_history.add_user_message(user_message)

    result = agent.invoke({
        "messages": chat_history.messages
    })

    messages = result["messages"]

    reasoning_trace = []

    for message in messages:

        # Agent decided to call a tool
        if hasattr(message, "tool_calls") and message.tool_calls:

            for tool_call in message.tool_calls:

                reasoning_trace.append({
                    "step": "tool_call",
                    "tool": tool_call["name"],
                    "arguments": tool_call["args"]
                })

        # Tool returned information
        if message.__class__.__name__ == "ToolMessage":

            reasoning_trace.append({
                "step": "tool_result",
                "tool": getattr(message, "name", None),
                "result": message.content
            })

    answer = messages[-1].content

    chat_history.add_ai_message(answer)

    return {
        "answer": answer,
        "reasoning": reasoning_trace
    }

In [68]:
def chat_and_reasoning(massage):
  result = chat(
      massage
  )

  print("ANSWER:")
  print(result["answer"])

  print("\nREASONING:")
  for step in result["reasoning"]:
      print(step)

بررسی edge cases (وقتای که اظلاعات کافی وجود ندارد)

In [69]:
chat_and_reasoning('چرا کارتم بعد از 10 روز هنوز نرسیده؟"')

ANSWER:
برای بررسی دقیق وضعیت کارت شما و دلیل تأخیر، ابتدا باید هویت شما را با کد ملی تأیید کنم. لطفاً کد ملی خود را وارد کنید تا بتوانم سفارش‌های فعال و تاریخچهٔ کارت‌های شما را بررسی کنم و原因 تأخیر را به شما بگویم.

REASONING:


نشان دادن چند سفارش

In [70]:
chat_and_reasoning('کد ملیم:7990475355 ')

7990475355
   national_id card_type  order_date          status tracking_code  \
0   7990475355      نقدی  1404/10/05    در دست بررسی     135712923   
24  7990475355      نقدی  1404/09/25       ارسال شده    1011520871   
26  7990475355   اعتباری  1404/09/21  تحویل داده شده    2182114766   

    estimated_delivery                                   shipping_address  
0   1404/10/19 (تأخیر)  تهران، یوسف آباد، خیابان رسالت، کوچه 26، پلاک ...  
24          1404/10/05  قم، فرمانیه، خیابان کارگر، کوچه 5، پلاک 238، و...  
26          1404/09/28  تهران، نیاوران، خیابان انقلاب، کوچه 3، پلاک 6،...  
[{'national_id': '7990475355', 'card_type': 'نقدی', 'order_date': '1404/10/05', 'status': 'در دست بررسی', 'tracking_code': '135712923', 'estimated_delivery': '1404/10/19 (تأخیر)', 'shipping_address': 'تهران، یوسف آباد، خیابان رسالت، کوچه 26، پلاک 216، واحد 18'}, {'national_id': '7990475355', 'card_type': 'نقدی', 'order_date': '1404/09/25', 'status': 'ارسال شده', 'tracking_code': '1011520871', 'estimat

In [71]:
chat_and_reasoning('کارتم رسید کد ملیم 7990475355?')

ANSWER:
حسین فرهادی عزیز، بر اساس اطلاعات سفارش‌های فعال شما:

- **سفارش اول** (کد پیگیری ۱۳۵۷۱۲۹۲۳) هنوز در مرحله **بررسی** است و تاریخ تحویل پیش‑بینی آن ۱۹ اکتبر ۱۴۰۴ (با تأخیر) است.  
- **سفارش دوم** (کد پیگیری ۱۰۱۱۵۲۰۸۷۱) **ارسال شده** و تاریخ تحویل پیش‑بینی آن ۵ اکتبر ۱۴۰۴ بود.

سیستم ما تنها تاریخ **تحویل پیش‑بینی** را نشان می‌دهد و تایید واقعی تحویل (مثلاً امضای دریافت‌کننده) را در دسترس ندارد. بنابراین نمی‌توانم با قطعی بگویم کارت شما به دست رسیده یا نه؛ اما اگر امروز تاریخ شما پس از ۵ اکتبر ۱۴۰۴ است، احتمال تحویل کارت دوم وجود دارد.

برای اطمینان کامل، می‌توانید با استفاده از کد پیگیری **۱۰۱۱۵۲۰۸۷۱** در سایت یا اپلیکیشن پست ایران وضعیت تحویل دقیق را بررسی کنید؛ اگر کارت هنوز در مسیر است یا در تحویل نهایی تأخیر داشته باشد، این اطلاعات هناك نمایش داده می‌شود.

سوال دیگری دارید یا می‌خواهید وضعیت را بعداً بررسی کنم؟

REASONING:


Request Logging
و ثبت درخواست های جدید



In [81]:
chat_and_reasoning(' میخوام یه کارت هدیه جدید بگیرم کد ملیم 1335175999 اگر میشه برای تاریخ 1405/5/5')

/tmp/ipykernel_3315/4223588388.py:253: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  request_df = pd.concat(


ANSWER:
درخواست شما برای دریافت **کارت هدیه جدید** با کد ملی **۱۳۳۵۱۷۵۹۹۹** (امیر حیدری) برای تاریخ **۱۴۰۵/۰۵/۰۵** با موفقیت در سیستم ثبت شد و در حال حاضر در وضعیت **„ثبت شده“** قرار دارد.

معمولاً زمان پردازش و تحویل کارت هدیه بین **۵ تا ۷ روز کاری** پس از تأیید درخواست است. می‌توانید در آینده با ارائه کد ملی وضعیت سفارش خود را بررسی کنید یا در صورت نیاز به اطلاعات تکمیلی با ما تماس بگیرید.

سوال دیگری دارید؟

REASONING:
{'step': 'tool_call', 'tool': 'authenticate_customer', 'arguments': {'national_id': '1335175999'}}
{'step': 'tool_result', 'tool': 'authenticate_customer', 'result': '{"authenticated": true, "national_id": "1335175999", "name": "امیر حیدری"}'}
{'step': 'tool_call', 'tool': 'create_request', 'arguments': {'national_id': '1335175999', 'request_type': 'سفارش کارت جدید', 'description': 'درخواست برای دریافت کارت هدیه جدید برای تاریخ 1405/5/5'}}
{'step': 'tool_result', 'tool': 'create_request', 'result': "{'success': True, 'duplicate': False, 'request': {'national_id': '133

In [83]:
print(db.requests_path)

/content/requests.xlsx


In [84]:
requests_df = pd.read_excel(
    db.requests_path
)

requests_df

,national_id,name,request_type,description,date,status
0,1335175999,امیر حیدری,سفارش کارت جدید,درخواست برای دریافت کارت هدیه جدید برای تاریخ ...,2026-10-06 18:40:30.557,ثبت شده
